In [0]:
base_path = "/Volumes/workspace/default/ecommerce_raw/"

orders_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(base_path + "olist_orders_dataset.csv")
)

In [0]:
(
    orders_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.orders_bronze")
)

In [0]:
spark.sql("""
    SELECT *
    FROM workspace.default.orders_bronze
    LIMIT 10
""").show()


In [0]:
spark.sql("""
    SELECT COUNT(*) AS total_rows
    FROM workspace.default.orders_bronze
""").show()

In [0]:
def ingest_to_bronze(file_name, table_name):

    file_path = base_path + file_name

    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(file_path)
    )

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"workspace.default.{table_name}")
    )

    print(f"Successfully created: {table_name}")

In [0]:
ingest_to_bronze(
    "olist_customers_dataset.csv",
    "customers_bronze"
)

In [0]:
spark.sql("""
    SELECT COUNT(*) AS total_rows
    FROM workspace.default.customers_bronze
""").show()

In [0]:
ingest_to_bronze(
    "olist_customers_dataset.csv",
    "customers_bronze"
)

ingest_to_bronze(
    "olist_order_items_dataset.csv",
    "order_items_bronze"
)

ingest_to_bronze(
    "olist_order_payments_dataset.csv",
    "payments_bronze"
)

ingest_to_bronze(
    "olist_order_reviews_dataset.csv",
    "reviews_bronze"
)

ingest_to_bronze(
    "olist_products_dataset.csv",
    "products_bronze"
)

ingest_to_bronze(
    "olist_sellers_dataset.csv",
    "sellers_bronze"
)

ingest_to_bronze(
    "olist_geolocation_dataset.csv",
    "geolocation_bronze"
)

ingest_to_bronze(
    "product_category_name_translation.csv",
    "category_translation_bronze"
)

In [0]:
tables = [
    "customers_bronze",
    "orders_bronze",
    "order_items_bronze",
    "payments_bronze",
    "reviews_bronze",
    "products_bronze",
    "sellers_bronze",
    "geolocation_bronze",
    "category_translation_bronze"
]

for table in tables:
    count = spark.table(f"workspace.default.{table}").count()
    print(f"{table}: {count}")

In [0]:
orders_bronze = spark.table("workspace.default.orders_bronze")
orders_bronze.printSchema()
orders_bronze.show(5, truncate=False)

In [0]:
reviews_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .option("multiLine", "true")
    .csv(
        base_path + "olist_order_reviews_dataset.csv"
    )
)

In [0]:
print("Reviews rows:", reviews_df.count())

In [0]:
reviews_df.printSchema()

In [0]:
from pyspark.sql.functions import col

bad_creation = reviews_df.filter(
    col("review_creation_date").isNotNull() &
    ~col("review_creation_date").rlike(
        r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$"
    )
)

bad_creation.select(
    "review_id",
    "order_id",
    "review_score",
    "review_comment_message",
    "review_creation_date",
    "review_answer_timestamp"
).show(30, truncate=False)

In [0]:
bad_answer = reviews_df.filter(
    col("review_answer_timestamp").isNotNull() &
    ~col("review_answer_timestamp").rlike(
        r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$"
    )
)

bad_answer.select(
    "review_id",
    "order_id",
    "review_score",
    "review_comment_message",
    "review_creation_date",
    "review_answer_timestamp"
).show(30, truncate=False)

In [0]:
reviews_df.select(
    "review_score",
    "review_comment_title",
    "review_comment_message",
    "review_creation_date",
    "review_answer_timestamp"
).show(20, truncate=False)

In [0]:
from pyspark.sql.functions import col

bad_creation = reviews_df.filter(
    col("review_creation_date").isNotNull() &
    ~col("review_creation_date").rlike(
        r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$"
    )
)

bad_answer = reviews_df.filter(
    col("review_answer_timestamp").isNotNull() &
    ~col("review_answer_timestamp").rlike(
        r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$"
    )
)

print("Bad creation dates:", bad_creation.count())
print("Bad answer timestamps:", bad_answer.count())

In [0]:
import csv

file_path = "/Volumes/workspace/default/ecommerce_raw/olist_order_reviews_dataset.csv"

rows = []

with open(file_path, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)

    for row in reader:
        rows.append(row)

print("Rows:", len(rows))

In [0]:
reviews_df = spark.createDataFrame(rows)

In [0]:
reviews_df.printSchema()

In [0]:
from pyspark.sql.functions import col

print(
    "Invalid review scores:",
    reviews_df.filter(
        ~col("review_score").isin("1", "2", "3", "4", "5")
        & col("review_score").isNotNull()
    ).count()
)

In [0]:
print(
    "Bad creation dates:",
    reviews_df.filter(
        col("review_creation_date").isNotNull() &
        ~col("review_creation_date").rlike(
            r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$"
        )
    ).count()
)

In [0]:
print(
    "Bad answer dates:",
    reviews_df.filter(
        col("review_answer_timestamp").isNotNull() &
        ~col("review_answer_timestamp").rlike(
            r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$"
        )
    ).count()
)

In [0]:
reviews_df.printSchema()

In [0]:
reviews_df.select(
    "review_score",
    "review_creation_date",
    "review_answer_timestamp"
).show(30, truncate=False)

In [0]:
(
    reviews_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "workspace.default.reviews_bronze"
    )
)

In [0]:
print(
    spark.table(
        "workspace.default.reviews_bronze"
    ).count()
)